In [ ]:
# Demo (PyTorch)

# Overview
- MNIST dataset
  + Goal: recognizing handwritten digits
  + Input: a 28x28 image
  + Output: class label (0, 1,..., 9)
- Defining an MLP for MNIST in PyTorch
- Training, evaluating, and extracting intermediate features (`fc2`)

# Import libraries

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
num_classes = 10   # 10 classes: 0, 1,..., 9

# (1) Data

In [ ]:
# Download and load MNIST via torchvision
train_raw = datasets.MNIST(root='./data', train=True, download=True)
test_raw = datasets.MNIST(root='./data', train=False, download=True)

x_train = train_raw.data.numpy().reshape(60000, 28, 28).astype('float32') / 255.0
y_train = train_raw.targets.numpy().astype('int64')

x_test = test_raw.data.numpy().reshape(10000, 28, 28).astype('float32') / 255.0
y_test = test_raw.targets.numpy().astype('int64')

print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

# Lưu ý: Trong PyTorch, nn.CrossEntropyLoss nhận trực tiếp nhãn số nguyên (0..9)
# nên không cần chuyển sang one-hot vector như keras.utils.to_categorical.

# (2) Declare model

In [ ]:
class MNIST_MLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.dropout1 = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(512, 512)
        self.dropout2 = nn.Dropout(p=0.2)
        self.fc3_10ways = nn.Linear(512, num_classes)

    def forward(self, x, return_fc2=False):
        x = self.flatten(x)
        x = self.dropout1(F.relu(self.fc1(x)))
        fc2_out = F.relu(self.fc2(x))
        x = self.dropout2(fc2_out)
        logits = self.fc3_10ways(x)
        if return_fc2:
            return fc2_out, logits
        return logits

model = MNIST_MLP(num_classes=num_classes).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable params: {trainable_params:,}')

In [ ]:
# Hiển thị bảng kiến trúc chi tiết (tương đương model.summary / plot_model)
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, 28, 28), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

# (3) Train defined model
- Training history (`loss`, `accuracy`, `val_loss`, `val_accuracy`) được lưu trong biến `history`
- Nếu gặp lỗi tràn VRAM (OOM), hãy giảm `batch_size`

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item() * bx.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == by).sum().item()
            total += bx.size(0)
    return total_loss / total, correct / total


# declare learning rate, loss function, optimizer
lr = 0.01
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

batch_size = 128
epochs = 30

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=batch_size, shuffle=False
)

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

starting_time = time.time()
for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(dim=1) == by).sum().item()
        total += bx.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(f'Epoch {epoch:2d}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}')

print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

# (4) Evaluate trained model

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

In [ ]:
# select a test image randomly
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]
test_label = int(y_test[random_test_index])

# predict test image with trained model
model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(test_img, axis=0)).to(device)
    probs = F.softmax(model(inp), dim=1).cpu().numpy()
pred_label = int(np.argmax(probs))

plt.imshow(test_img, cmap='gray')
plt.title('true label = %d, predicted label = %d' % (test_label, pred_label))
plt.show()

In [ ]:
# Kiểm tra chi tiết kích thước & xác suất dự đoán (tương đương các cell cuối bản Keras)
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]
test_label = int(y_test[random_test_index])

plt.imshow(test_img, cmap='gray')
plt.show()
print('test_label:', test_label)
print('test_img.shape:', test_img.shape)
print('expanded shape:', np.expand_dims(test_img, axis=0).shape)

model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(test_img, axis=0)).to(device)
    y_pred = F.softmax(model(inp), dim=1).cpu().numpy()

print('y_pred.shape:', y_pred.shape)
print('sum(y_pred):', np.sum(y_pred, axis=1))
print('argmax(y_pred):', np.argmax(y_pred, axis=1))

### Trích xuất đặc trưng tầng ẩn `fc2` (tương đương `Model(inputs=model.input, outputs=[model.get_layer('fc2').output, model.output])`)

In [ ]:
model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(test_img, axis=0)).to(device)
    fc2_feat, logits = model(inp, return_fc2=True)
    y_feat = fc2_feat.cpu().numpy()
    y_pred = F.softmax(logits, dim=1).cpu().numpy()

print('y_feat.shape, y_pred.shape:', y_feat.shape, y_pred.shape)
print('First 20 values of y_feat:', y_feat[0, :20])